Python script implements the Tree of Thoughts (ToT) prompting framework using the OpenAI API to systematically explore and solve complex problems. It uses a Breadth-First Search (BFS) approach with two main phases per iteration:

    Generation (explore_thoughts): It takes current ideas and prompts the LLM with a creative temperature (0.7) to brainstorm and branch out multiple new reasoning paths.

    Evaluation/Pruning (evaluate_thoughts): It acts as a strict judge, prompting the LLM with a deterministic temperature (0.0) to assess the viability of the new ideas. Invalid thoughts are pruned, and only approved ideas survive.

Ultimately, instead of generating a single linear response, the code outputs a tree structure containing multiple validated, high-quality solutions.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"), override=True)

True

In [2]:
class OpenAIService:

    def __init__(self, model_name: str = "gpt-4o-mini", api_key: str = None):        
        self.client = OpenAI()
        self.model_name = model_name

    def generate_response(self, prompt: str, max_tokens: int = 1024, temperature: float = 0.7) -> str:
        response = self.client.chat.completions.create(
            model=self.model_name,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

In [3]:
class ThoughtNode:
    
    def __init__(self, thought, children=None, status="PENDING"):
        self.thought = thought
        self.children = children or []
        self.status = status  # Can be: PENDING, APPROVED, DISCARDED

In [4]:
class TreeOfThought:
    
    def __init__(self, root_prompt, ai_service=None, max_iterations=3, max_tokens=250):
        self.root = ThoughtNode(root_prompt, status="ROOT")
        self.max_iterations = max_iterations
        self.ai_service = ai_service or OpenAIService()
        self.current_thoughts = [self.root]
        self.max_tokens = max_tokens

    def call_llm(self, prompt, temperature=0.7):
        response = self.ai_service.generate_response(
            prompt,
            max_tokens=self.max_tokens,
            temperature=temperature
        )
        return response

    # Generator of ideas or reasoning
    def explore_thoughts(self, thought_nodes):
        """Generates new branches from the current thoughts."""
        new_thought_nodes = []
        for thought_node in thought_nodes:
            prompt = f"Given the current thought: '{thought_node.thought}', provide two concise next thoughts that evolve this idea further. Separate them clearly."
            
            # We use a bit of temperature (0.7) for more creativity in generation
            response = self.call_llm(prompt, temperature=0.7)
            
            if response:
                # Note: Ideally it would be requested in JSON to separate easily. 
                # Here we store it as a single node for simplicity, but you can 
                # improve the prompt to split the exact response.
                new_thought_node = ThoughtNode(response)
                thought_node.children.append(new_thought_node)
                new_thought_nodes.append(new_thought_node)
                
        return new_thought_nodes

    # Evaluator
    def evaluate_thoughts(self, thought_nodes):
        """Evaluates generated ideas and filters out those that are not viable (Pruning)."""
        surviving_nodes = []
        
        for node in thought_nodes:
            eval_prompt = f"""
            Evaluate the viability, logic, and impact of the following idea or thought: 
            '{node.thought}'
            
            Respond ONLY with a single word from the following list to rate it:
            - APPROVED (If the idea makes sense, is logical, and deserves further exploration)
            - DISCARDED (If the idea does not make sense, is repetitive, illogical, or unfeasible)
            """
            
            # We use low temperature (0.0) because we want an analytical and deterministic evaluation
            evaluation = self.call_llm(eval_prompt, temperature=0.0)
            
            if evaluation:
                evaluation_clean = evaluation.strip().upper()
                
                # If the evaluation contains the word "APPROVED", it survives
                if "APPROVED" in evaluation_clean:
                    node.status = "APPROVED"
                    surviving_nodes.append(node)
                else:
                    node.status = "DISCARDED"
                    print(f"  [PRUNING] The thought was discarded because it was evaluated as unfeasible.")
            else:
                # In case of API error, we keep it as a precaution
                node.status = "APPROVED (Evaluation error)"
                surviving_nodes.append(node)
                
        return surviving_nodes

    def run(self):
        iteration = 0
        while self.current_thoughts and iteration < self.max_iterations:
            print(f"\n{'='*20} Iteration {iteration + 1} {'='*20}")
            
            # 1. Generation phase
            print("Generating new ideas...")
            new_candidates = self.explore_thoughts(self.current_thoughts)
            
            # 2. Evaluation phase (Evaluator)
            print("Evaluating candidates with the evaluator...")
            self.current_thoughts = self.evaluate_thoughts(new_candidates)
            
            # Show survivors
            print(f"\n{len(self.current_thoughts)} thought(s) survived the evaluation:")
            for thought_node in self.current_thoughts:
                # Print an extract to avoid flooding the console
                extract = thought_node.thought.replace('\n', ' ')[:80]
                print(f"  -> {extract}...")
                
            iteration += 1

    def update_starting_thought(self, new_thought):
        self.root = ThoughtNode(new_thought, status="ROOT")
        self.current_thoughts = [self.root]

    def print_tree(self, node, level=0):
        """Prints the tree showing which branches were pruned."""
        indent = ' ' * (level * 4)
        status_mark = "✅" if node.status in ["APPROVED", "ROOT"] else "❌"
        status_mark = "⏳" if node.status == "PENDING" else status_mark
        
        # We take only the first line of the thought for tree visualization
        first_line = node.thought.split('\n')[0][:60] + "..."
        print(f"{indent}{status_mark} [{node.status}] {first_line}")
        
        for child in node.children:
            self.print_tree(child, level + 1)

In [5]:
starting_prompt = "Think of a solution to reduce the operational costs of your business."
    
# We create and initialize the tree
tot = TreeOfThought(starting_prompt, max_iterations=3)
tot.run()

print("\n" + "=" * 60)
print("FINAL THOUGHT TREE (Green branches survived, red crosses were pruned):")
tot.print_tree(tot.root)


==================== Iteration 1 ====================
Generating new ideas...
Evaluating candidates with the evaluator...

1 thought(s) survived the evaluation:
  -> 1. **Implement Technology Automation:** Explore the integration of automation to...

==================== Iteration 2 ====================
Generating new ideas...
Evaluating candidates with the evaluator...

1 thought(s) survived the evaluation:
  -> 3. **Enhance Data Analytics Capabilities:** Invest in advanced data analytics to...

==================== Iteration 3 ====================
Generating new ideas...
Evaluating candidates with the evaluator...

1 thought(s) survived the evaluation:
  -> 5. **Integrate Artificial Intelligence and Machine Learning:** Utilize AI and ma...

FINAL THOUGHT TREE (Green branches survived, red crosses were pruned):
✅ [ROOT] Think of a solution to reduce the operational costs of your ...
    ✅ [APPROVED] 1. **Implement Technology Automation:** Explore the integrat...
        ✅ [APPROVED] 